In [1]:
# Sel 1: Import Library
import os
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Sel 2: Konfigurasi Direktori
SPLIT_TRAIN_DIR = '../data/split/train/'   # <-- sumber augmentasi
SPLIT_TEST_DIR = '../data/split/test/'     # <-- sumber test murni

AUG_TRAIN_DIR = '../data/augmented/train/'      # <-- output: train hasil augmentasi
AUG_TEST_DIR = '../data/augmented/test/'        # <-- output: test, disalin tanpa augmentasi

CLASSES = ['healthy', 'sick']

In [3]:
# Sel 3: Siapkan kerangka folder output
for class_name in CLASSES:
    os.makedirs(os.path.join(AUG_TRAIN_DIR, class_name), exist_ok=True)
    os.makedirs(os.path.join(AUG_TEST_DIR, class_name), exist_ok=True)
    print(f"Folder disiapkan: {os.path.join(AUG_TRAIN_DIR, class_name)}")
    print(f"Folder disiapkan: {os.path.join(AUG_TEST_DIR, class_name)}")

Folder disiapkan: ../data/augmented/train/healthy
Folder disiapkan: ../data/augmented/test/healthy
Folder disiapkan: ../data/augmented/train/sick
Folder disiapkan: ../data/augmented/test/sick


In [4]:
# Sel 4: Fungsi Implementasi 6 Teknik Augmentasi Jurnal
def apply_augmentations_and_save(image_path, filename, save_dir):
    # Baca gambar menggunakan OpenCV
    img = cv2.imread(image_path)
    if img is None:
        return 0

    # 1. Simpan Original
    cv2.imwrite(os.path.join(save_dir, f"orig_{filename}"), img)

    # 2. Flip Horizontal
    flip_h = cv2.flip(img, 1)
    cv2.imwrite(os.path.join(save_dir, f"fliph_{filename}"), flip_h)

    # 3. Flip Vertikal
    flip_v = cv2.flip(img, 0)
    cv2.imwrite(os.path.join(save_dir, f"flipv_{filename}"), flip_v)

    # 4. Rotate 90 Derajat
    rot_90 = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    cv2.imwrite(os.path.join(save_dir, f"rot90_{filename}"), rot_90)

    # 5. Rotate 180 Derajat
    rot_180 = cv2.rotate(img, cv2.ROTATE_180)
    cv2.imwrite(os.path.join(save_dir, f"rot180_{filename}"), rot_180)

    # 6. Adjust Brightness (Meningkatkan Kecerahan)
    # Konversi ke HSV untuk memanipulasi Value (kecerahan) tanpa merusak warna
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    v = cv2.add(v, 40) # Tambah intensitas cahaya
    bright_img = cv2.cvtColor(cv2.merge((h, s, v)), cv2.COLOR_HSV2BGR)
    cv2.imwrite(os.path.join(save_dir, f"bright_{filename}"), bright_img)

    # 7. Adjust Contrast (Meningkatkan Kontras)
    # Menggunakan alpha (gain) > 1 untuk melebarkan jarak antar piksel
    contrast_img = cv2.convertScaleAbs(img, alpha=1.5, beta=0)
    cv2.imwrite(os.path.join(save_dir, f"contrast_{filename}"), contrast_img)

    return 7 # Mengembalikan jumlah gambar yang dihasilkan

In [5]:
# Sel 5: Eksekusi Augmentasi pada data/split/train/
print("Memulai proses augmentasi pada TRAIN split saja...")
total_augmented = 0

for class_name in CLASSES:
    class_train_dir = os.path.join(SPLIT_TRAIN_DIR, class_name)
    class_aug_dir = os.path.join(AUG_TRAIN_DIR, class_name)

    count = 0
    for img_name in os.listdir(class_train_dir):
        img_path = os.path.join(class_train_dir, img_name)
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            generated = apply_augmentations_and_save(img_path, img_name, class_aug_dir)
            count += generated

    total_augmented += count
    print(f"Kelas '{class_name}' (train): {count} gambar berhasil di-generate.")

print(f"\nSelesai! Total dataset TRAIN setelah augmentasi: {total_augmented} gambar.")

Memulai proses augmentasi pada TRAIN split saja...
Kelas 'healthy' (train): 560 gambar berhasil di-generate.
Kelas 'sick' (train): 560 gambar berhasil di-generate.

Selesai! Total dataset TRAIN setelah augmentasi: 1120 gambar.


In [6]:
# Sel 6: Salin data/split/test/ -> data/augmented/test/
print("Menyalin TEST split tanpa augmentasi...")
total_test = 0

for class_name in CLASSES:
    class_test_src = os.path.join(SPLIT_TEST_DIR, class_name)
    class_test_dst = os.path.join(AUG_TEST_DIR, class_name)

    count = 0
    for img_name in os.listdir(class_test_src):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy2(os.path.join(class_test_src, img_name),
                         os.path.join(class_test_dst, img_name))
            count += 1

    total_test += count
    print(f"Kelas '{class_name}' (test): {count} gambar disalin apa adanya.")

print(f"\nTotal dataset TEST (murni, tanpa augmentasi): {total_test} gambar.")
print("\nRingkasan akhir:")
print(f"  Train (augmented): {total_augmented} gambar -> '{AUG_TRAIN_DIR}'")
print(f"  Test  (murni)    : {total_test} gambar -> '{AUG_TEST_DIR}'")

Menyalin TEST split tanpa augmentasi...
Kelas 'healthy' (test): 20 gambar disalin apa adanya.
Kelas 'sick' (test): 20 gambar disalin apa adanya.

Total dataset TEST (murni, tanpa augmentasi): 40 gambar.

Ringkasan akhir:
  Train (augmented): 1120 gambar -> '../data/augmented/train/'
  Test  (murni)    : 40 gambar -> '../data/augmented/test/'
